# **ML Model:** Logistic Regression

## **Notes:**

**Logistic regression** model will serve as **baseline model** in this project.

His purpose is to provide **reference point** which will be used to evaluate **advanced models**. It is commonly used for this because of its efficiency, simplicity, strong performance on many classification task.

Results that are obtained here will later be compared with more advanced (complex) models in this project:
- **Random Forest** (RF)
- **eXtreme Gradient Boosting** (XGBoost)

## **Implementation:**

#### Librabry imports

In [24]:
import numpy as np
import pandas as pd
import category_encoders as ce
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression

### **Load data**

In [2]:
data = pd.read_excel('../data/processed/accident_processed_srb.xlsx')

In [ ]:
def load_info(df):
    print(f'Shape: {data.shape}')
    print(f'Columns: {data.columns.to_list()}')
    print(f'Number of coumnts: {len(data.columns)}')
    print(f'Target distribution: {round(data['accident_type'].value_counts(normalize=True), 3)}')

In [ ]:
load_info(df=data)

Shape: (205352, 16)
Columns: ['municipality', 'longitude', 'latitude', 'accident_type', 'month', 'day_of_week', 'hour', 'day_type', 'is_rush', 'is_night', 'season', 'acc_parked_vehicles', 'acc_pedestrians', 'acc_single_vehicle', 'acc_two_vehicles_no_turn', 'acc_two_vehicles_turn_or_cross']
Number of coumnts: 16
Target distribution: accident_type
0    0.597
1    0.403
Name: proportion, dtype: float64


In [14]:
y = data['accident_type']
X = data.drop(columns=['accident_type'])

### **Split data**

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
def split_info(df):
    print(f'Rows number (Train): {X_train.shape[0]}')
    print(f'Rows number (Test): {X_test.shape[0]}')
    print(f'Target distribution (Train): {round(y_train.value_counts(normalize=True), 3).to_dict()}')
    print(f'Target distribution (Test): {round(y_test.value_counts(normalize=True), 3).to_dict()}')

In [21]:
split_info(df=data)

Rows number (Train): 164281
Rows number (Test): 41071
Target distribution (Train): {0: 0.597, 1: 0.403}
Target distribution (Test): {0: 0.597, 1: 0.403}


#### Target Encoding

In [28]:
mun_encoder = ce.TargetEncoder(cols=['municipality'])

In [29]:
X_train['municipality'] = mun_encoder.fit_transform(X_train['municipality'], y_train)
X_test['municipality'] = mun_encoder.transform(X_test['municipality'])

#### Scaling features

Each feature gets value of mean set to 0 and value of std. set to 1.

In [25]:
sc = StandardScaler()

##### Data before scaling

In [36]:
X_train.head()

,municipality,longitude,latitude,month,day_of_week,hour,day_type,is_rush,is_night,season,acc_parked_vehicles,acc_pedestrians,acc_single_vehicle,acc_two_vehicles_no_turn,acc_two_vehicles_turn_or_cross
175803,0.266285,20.393045,44.840774,10,4,13,1,0,0,3,0,0,0,1,0
100351,0.239874,20.404830,44.814717,2,6,13,0,0,0,0,0,0,0,0,1
167577,0.424613,20.241943,44.662004,9,1,20,1,0,0,3,0,0,0,0,1
89852,0.613055,20.644826,44.871233,5,2,17,1,1,0,1,0,0,0,1,0
127493,0.566962,19.651931,46.124863,9,1,12,1,0,0,3,0,0,0,0,1


In [37]:
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

##### Data after scaling

In [38]:
X_train_sc_prev = pd.DataFrame(X_test_sc, columns=X_train.columns)
X_train_sc_prev.head()

,municipality,longitude,latitude,month,day_of_week,hour,day_type,is_rush,is_night,season,acc_parked_vehicles,acc_pedestrians,acc_single_vehicle,acc_two_vehicles_no_turn,acc_two_vehicles_turn_or_cross
0,-1.256728,-0.097851,-0.043303,-1.027986,-1.498173,-0.788266,0.600318,-0.607301,-0.459982,-0.456405,-0.422156,3.451395,-0.536716,-0.661832,-0.566654
1,-1.358207,-0.097851,-0.043303,1.563430,1.046974,0.105250,-1.665785,-0.607301,-0.459982,-1.343518,-0.422156,-0.289738,-0.536716,-0.661832,1.764746
2,0.725591,-0.097851,-0.043303,-0.164181,-0.989143,-1.145673,0.600318,1.646630,-0.459982,0.430707,-0.422156,-0.289738,-0.536716,-0.661832,1.764746
3,0.820940,-0.097851,-0.043303,0.699625,0.028915,-1.145673,0.600318,1.646630,-0.459982,1.317819,2.368791,-0.289738,-0.536716,-0.661832,-0.566654
4,-1.169594,-0.097851,-0.043303,0.411690,-0.480114,-0.073453,0.600318,-0.607301,-0.459982,0.430707,-0.422156,-0.289738,-0.536716,1.510958,-0.566654


In [39]:
X_train_sc

array([[-0.93225793, -0.09785062, -0.04330282, ..., -0.53671615,
         1.51095784, -0.56665383],
       [-1.11206052, -0.09785062, -0.04330282, ..., -0.53671615,
        -0.66183183,  1.76474585],
       [ 0.1456203 , -0.09785062, -0.04330282, ..., -0.53671615,
        -0.66183183,  1.76474585],
       ...,
       [ 1.78652126, -0.09785062, -0.04330282, ..., -0.53671615,
        -0.66183183, -0.56665383],
       [-1.25672793, -0.09785062, -0.04330282, ..., -0.53671615,
        -0.66183183,  1.76474585],
       [-1.11971521, -0.09785062, -0.04330282, ..., -0.53671615,
         1.51095784, -0.56665383]], shape=(164281, 15))

### **Training**

### **Evaluation**

### **Additional:**